# Staged Prediction of Postoperative Complications After Esophageal Cancer Surgery
## DUCA Registry 2016-2025

---

We build the prediction model in **5 progressive stages**, adding predictor groups one at a time to show how much each domain contributes to accuracy.

| Stage | Predictor group added | Clinical rationale |
|---|---|---|
| 1 | Demographics + fitness | What you know before the patient even enters clinic |
| 2 | + Comorbidities + lifestyle | Patient-level risk burden |
| 3 | + Tumor + neoadjuvant treatment | Disease severity and treatment effect |
| 4 | + Surgical factors | Intra-operative events (blood loss, duration, conversion) |
| 5 | + Pulmonary function (fev1vc) | Sensitivity analysis — subset with spirometry |

**Primary outcome**: `compl` (any complication, yes/no) — 1,181 patients, 57.6% event rate.

**Secondary outcomes** (Stage 4 best model): `comppulm`, `compcar`, `compand` (pulmonary, cardiac, anastomotic leak).

**Models per stage**: Logistic Regression (L1/LASSO) + Random Forest + XGBoost, 5-fold stratified CV.

**Metrics**: AUC-ROC, balanced accuracy, sensitivity, specificity, calibration plot.

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score, roc_curve,
    confusion_matrix, classification_report, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except ImportError:
    print("xgboost not installed — run: pip install xgboost")
    XGB_AVAILABLE = False

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    print("shap not installed — run: pip install shap")
    SHAP_AVAILABLE = False

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')

RANDOM_STATE = 42
N_FOLDS = 5

# Source data is not included in this public repo (patient-level registry data).
# Place the raw DUCA export at the path below (relative to this notebook) to reproduce.
FILE_PATH = "../data/duca_esophagectomy_raw.xlsx"

print("Setup complete.")

---
## 1. Load & Merge Data

In [ ]:
df1 = pd.read_excel(FILE_PATH, sheet_name=0)
df2 = pd.read_excel(FILE_PATH, sheet_name=1)

def pad_upn(df, col='upn', digits=7):
    df = df.copy()
    df[col] = df[col].astype(str).str.strip().str.split('.').str[0].str.zfill(digits)
    return df

df1 = pad_upn(df1)
df2 = pad_upn(df2)
df  = df1.merge(df2, on='upn', how='left')

# ── Fix _x / _y merge duplicates ─────────────────────────────────────────────
# When both sheets share a column name, pandas appends _x (sheet1) and _y (sheet2).
# We keep the sheet1 version (_x) and rename it back to the base name.
x_cols = [c for c in df.columns if c.endswith('_x')]
for col_x in x_cols:
    base  = col_x[:-2]
    col_y = base + '_y'
    if col_y in df.columns:
        df.rename(columns={col_x: base}, inplace=True)
        df.drop(columns=[col_y], inplace=True)
        print(f"  Duplicate column resolved: kept '{base}' (sheet 1), dropped '{col_y}' (sheet 2)")

print(f"\nMerged dataset: {df.shape[0]} patients, {df.shape[1]} columns")

---
## 2. Preprocessing Pipeline

All decisions follow the Statistical Analysis Plan (SAP, 27 May 2026).

In [ ]:
# ── 2.1  Replace sentinel missing values ──────────────────────────────────────
df.replace(['##USER_MISSING_99##', 'Onbekend'], np.nan, inplace=True)

# Clavien-Dindo grade columns: 9/99/999 = unknown
clav_cols = [c for c in df.columns if c.startswith('clav')]
for c in clav_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').replace({9: np.nan, 99: np.nan, 999: np.nan})

# Column-specific sentinel cleaning
# Each sentinel value is a registry code for "unknown" — replace with NaN
sentinel_map = {
    'asascore':  [9],        # ASA 1–5; 9 = not recorded
    'scorect':   [99],       # T-stage; 99 = unknown
    'scorecn':   [9],        # N-stage; 9 = unknown
    'neoadjaf':  [9],        # neoadjuvant response; 9 = unknown
    'curaok':    [9],        # curative surgery (0/1); 9 = unknown
}
for col, vals in sentinel_map.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        for v in vals:
            n = (df[col] == v).sum()
            if n > 0:
                df.loc[df[col] == v, col] = np.nan
                print(f"  {col}: {n} sentinel value(s) of {v} → NaN")

# ── 2.2  Derived variables ────────────────────────────────────────────────────
# Age at surgery
if 'datok' in df.columns and 'gebdat' in df.columns:
    datok_dt  = pd.to_datetime(df['datok'],  errors='coerce', dayfirst=True)
    gebdat_dt = pd.to_datetime(df['gebdat'], errors='coerce', dayfirst=True)
    if datok_dt.isna().mean() > 0.5:
        datok_dt  = pd.to_datetime(df['datok'],  unit='D', origin='1899-12-30', errors='coerce')
    if gebdat_dt.isna().mean() > 0.5:
        gebdat_dt = pd.to_datetime(df['gebdat'], unit='D', origin='1899-12-30', errors='coerce')
    df['age_at_op'] = (datok_dt - gebdat_dt).dt.days // 365
    print(f"age_at_op: {df['age_at_op'].notna().sum()} valid  |  "
          f"range {df['age_at_op'].min():.0f}–{df['age_at_op'].max():.0f} yrs")

# BMI (prefer existing column; compute if absent)
if 'BMI' not in df.columns:
    if 'gewicht' in df.columns and 'lengte' in df.columns:
        g = pd.to_numeric(df['gewicht'], errors='coerce')
        l = pd.to_numeric(df['lengte'],  errors='coerce')
        df['BMI'] = g / (l / 100) ** 2
        print(f"BMI computed: {df['BMI'].notna().sum()} valid")

# Mortality flag from dataovl  (NaN = survived)
if 'dataovl' in df.columns:
    df['mort'] = pd.to_numeric(df['dataovl'], errors='coerce').notna().astype(int)
    print(f"mort (death): {df['mort'].sum()} events")

# Clavien-Dindo derived binary outcome
if clav_cols:
    df['derived_clavien'] = df[clav_cols].max(axis=1)
    # Minor: grade I, II, IIIa (1-3) | Major: grade IIIb, IVa, IVb, V (4-7)
    df['derived_clavien_bin'] = np.where(
        df['derived_clavien'].between(1, 3), 0,
        np.where(df['derived_clavien'].between(4, 7), 1, np.nan)
    )
    print(f"Clavien-Dindo binary (cutoff IIIb+) — minor (I–IIIa): {(df['derived_clavien_bin']==0).sum()}  "
          f"major (IIIb+): {(df['derived_clavien_bin']==1).sum()}")

# ── 2.3  ICU admission binary (from icdg) ─────────────────────────────────────
# icdg = ICU days. 999 is a registry sentinel for unknown — clean it first.
# Then create binary: did the patient spend ANY time in ICU? (>0 days)
if 'icdg' in df.columns:
    df['icdg'] = pd.to_numeric(df['icdg'], errors='coerce').replace(999, np.nan)
    df['icu_admission'] = (df['icdg'] > 0).astype(float)
    df.loc[df['icdg'].isna(), 'icu_admission'] = np.nan
    n_icu = int(df['icu_admission'].sum())
    print(f"icu_admission: {n_icu} patients ({n_icu/df['icu_admission'].notna().sum()*100:.1f}%) "
          f"had any ICU stay")

# ── 2.4  Readmission (heropn) — coerce to binary 0/1 ──────────────────────────
if 'heropn' in df.columns:
    heropn_num = pd.to_numeric(df['heropn'], errors='coerce')
    df['heropn'] = np.where(heropn_num.isna(), np.nan, (heropn_num > 0).astype(float))
    n_heropn = int(df['heropn'].sum())
    print(f"heropn (readmission): {n_heropn} events ({n_heropn/df['heropn'].notna().sum()*100:.1f}%)")

print("\nDerived variables done.")

In [ ]:
# ── 2.3  Encode Ja/Nee columns to 1/0 ────────────────────────────────────────
ja_nee_map = {'Ja': 1, 'ja': 1, 'Nee': 0, 'nee': 0}
for col in df.columns:
    if df[col].dtype == object:
        uniq = set(df[col].dropna().unique())
        if uniq <= {'Ja', 'Nee', 'ja', 'nee'}:
            df[col] = df[col].map(ja_nee_map)

# ── 2.4  Smoking — encode roken, add missing indicator ────────────────────────
if 'roken' in df.columns:
    roken_map = {
        'Nooit gerookt': 0, 'nooit gerookt': 0, 'Never': 0,
        'Gestopt':       1, 'gestopt':       1, 'Former': 1,
        'Momenteel':     2, 'momenteel':     2, 'Current': 2,
    }
    df['roken'] = df['roken'].map(roken_map)
    df['roken_missing'] = df['roken'].isna().astype(int)
    n_miss = df['roken_missing'].sum()
    print(f"roken after encoding:")
    print(df['roken'].value_counts(dropna=False).to_string())
    print(f"roken_missing indicator: {n_miss} patients ({n_miss/len(df)*100:.1f}%) had unknown smoking status")

# ── 2.5  Missing indicators for key surgical variables (BEFORE log-transform) ─
# Blood loss (32% missing) and op duration (19.7% missing) are top SHAP predictors.
# Missingness is likely informative (e.g. not recorded when blood loss was minimal).
# We add a binary flag so the model can learn the signal in "not recorded".
for col, flag in [('bloedverlies', 'bloedverlies_missing'),
                  ('duur_operatie', 'duur_operatie_missing')]:
    if col in df.columns:
        df[flag] = df[col].isna().astype(int)
        n = df[flag].sum()
        print(f"{flag}: {n} patients ({n/len(df)*100:.1f}%) flagged")

# ── 2.6  Log-transform right-skewed continuous variables ──────────────────────
for col in ['bloedverlies', 'duur_operatie', 'gewverl']:
    if col in df.columns:
        df[col] = np.log1p(pd.to_numeric(df[col], errors='coerce').clip(lower=0))

# ── 2.7  Packyears — sentinel cleaning + never-smoker logical deduction ───────
if 'packyears' in df.columns:
    df['packyears'] = pd.to_numeric(df['packyears'], errors='coerce')
    for val in [-96, 999, 100, 106]:
        n = (df['packyears'] == val).sum()
        if n > 0:
            df.loc[df['packyears'] == val, 'packyears'] = np.nan
            print(f"\npackyears: {n} sentinel values of {val} → NaN")
    if 'roken' in df.columns:
        never_n = ((df['roken'] == 0) & df['packyears'].isna()).sum()
        df.loc[(df['roken'] == 0) & df['packyears'].isna(), 'packyears'] = 0
        print(f"packyears: {never_n} confirmed never-smokers → 0 (logical, not imputed)")
    print(f"packyears remaining NaN: {df['packyears'].isna().sum()} — excluded from model (>50% missing)")

# ── 2.8  neoadjsch — logical fill before checking missingness ─────────────────
if 'neoadjsch' in df.columns and 'neoadj' in df.columns:
    df['neoadjsch'] = pd.to_numeric(df['neoadjsch'], errors='coerce')
    df['neoadj']    = pd.to_numeric(df['neoadj'],    errors='coerce')

    logical_fill_n = ((df['neoadj'] == 0) & df['neoadjsch'].isna()).sum()
    df.loc[(df['neoadj'] == 0) & df['neoadjsch'].isna(), 'neoadjsch'] = 0

    remaining_na  = df['neoadjsch'].isna().sum()
    remaining_pct = remaining_na / len(df) * 100
    print(f"\nneoadjsch:")
    print(f"  No-treatment patients filled with 0: {logical_fill_n}")
    print(f"  Remaining NaN (truly unknown cycles): {remaining_na} ({remaining_pct:.1f}%)")
    if remaining_pct > 50:
        print(f"  → EXCLUDE from model (>{50}% missing after logical fill)")
    else:
        print(f"  → KEEP in model, remaining NaN median-imputed in prepare_xy")

print("\nEncoding done.")

In [ ]:
# ── 2.8  Charlson Comorbidity Index (CCI) ─────────────────────────────────────
# Per 27 May meeting: combine comorbidities into clinical groups to reduce
# sparsity (Marieke suggestion). CCI is the standard validated index.
# comaf + comcarritme kept separately — most common cardiac events post-esophagectomy,
# not in standard CCI but clinically critical for this population.

CCI_WEIGHTS = {
    'commyo':      1,  # myocardial infarction
    'comhartfaal': 1,  # congestive heart failure
    'comperif':    1,  # peripheral vascular disease
    'comcva':      1,  # cerebrovascular disease (stroke)
    'comlong':     1,  # COPD / lung disease
    'comgiulcus':  1,  # peptic ulcer disease
    'comdiam':     1,  # diabetes (uncomplicated)
    'comnier':     2,  # renal disease
    'commalig':    2,  # prior malignancy
}

cci_available = [c for c in CCI_WEIGHTS if c in df.columns]
for c in cci_available:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)

df['CCI'] = sum(df[c] * CCI_WEIGHTS[c] for c in cci_available)

print(f"CCI computed from {len(cci_available)}/{len(CCI_WEIGHTS)} comorbidity variables")
print(f"  Range: {df['CCI'].min():.0f} – {df['CCI'].max():.0f}")
print(f"  Mean:  {df['CCI'].mean():.2f}   Median: {df['CCI'].median():.0f}")
print(f"\nCCI score distribution:")
print(df['CCI'].value_counts().sort_index().to_string())

# ── 2.9  Complication type categorical outcome ─────────────────────────────────
# Per 27 May meeting: "complication type as categorical: surgical, leakage, pulmonary"
# Priority when multiple types co-occur: leakage > pulmonary > cardiac > other
comp_type_map = {
    'compand':  2,   # anastomotic leakage  — highest clinical severity
    'comppulm': 1,   # pulmonary
    'compcar':  3,   # cardiac
}
df['comp_type'] = 0  # 0 = no complication
if 'compl' in df.columns:
    df.loc[pd.to_numeric(df['compl'], errors='coerce') == 1, 'comp_type'] = 4  # other
for col, val in sorted(comp_type_map.items(), key=lambda x: -x[1]):
    if col in df.columns:
        mask = pd.to_numeric(df[col], errors='coerce') == 1
        df.loc[mask, 'comp_type'] = val

labels = {0:'None', 1:'Pulmonary', 2:'Anastomotic leak', 3:'Cardiac', 4:'Other'}
print(f"\nComplication type distribution:")
print(df['comp_type'].map(labels).value_counts().to_string())

---
## 3. Outcome Variables

In [ ]:
# Check available outcomes and their event counts
outcomes = {
    'compl':       'Any complication',
    'comppulm':    'Pulmonary',
    'compcar':     'Cardiac',
    'compand':     'Anastomotic leak',
    'compinfect':  'Infection',
    'compwond':    'Wound',
    'compthrom':   'Thrombosis',
    'compgas':     'Gastrointestinal',
    'compchylek':  'Chyle leak',
    'compuro':     'Urological',
    'compneu':     'Neurological',
}

print(f"{'Outcome':<25} {'N events':>10} {'% events':>10} {'Modelable?':>12}")
print("-" * 62)
for col, label in outcomes.items():
    if col in df.columns:
        n = int(pd.to_numeric(df[col], errors='coerce').sum())
        pct = pd.to_numeric(df[col], errors='coerce').mean() * 100
        ok  = 'YES' if n >= 100 else f'LOW ({n})'
        print(f"{label:<25} {n:>10} {pct:>9.1f}% {ok:>12}")

print("\nPrimary outcome for staged models: compl (any complication)")
print("Secondary outcomes (applied to Stage 4 best model): comppulm, compcar, compand")

In [ ]:
OHE_COLS = ['tumlok1', 'tumhist', 'tumkeus', 'neoadjtype',
            'typok', 'aardok', 'procok', 'recontype', 'analig',
            'rasecok', 'reslmm']

ohe_present = [c for c in OHE_COLS if c in df.columns]
df_encoded  = pd.get_dummies(df, columns=ohe_present, drop_first=True, dummy_na=False)

new_dummies = [c for c in df_encoded.columns if c not in df.columns]
sparse      = [c for c in new_dummies if df_encoded[c].sum() < 20]
df_encoded.drop(columns=sparse, inplace=True)
print(f"OHE done — {df_encoded.shape[1]} columns, {len(sparse)} sparse dummies dropped")

# Sanity check: any object-dtype columns remaining?
obj_remaining = [c for c in df_encoded.columns if df_encoded[c].dtype == object]
if obj_remaining:
    print(f"WARNING: {len(obj_remaining)} object columns still present — add to OHE_COLS: {obj_remaining[:5]}")
else:
    print("All columns numeric after OHE.")

In [ ]:
def get_existing(cols):
    return [c for c in cols if c in df_encoded.columns]

def get_ohe_cols(prefix):
    return [c for c in df_encoded.columns if c.startswith(prefix + '_')]

# ── Stage 1: Demographics & Fitness ──────────────────────────────────────────
STAGE1 = get_existing(['age_at_op', 'geslacht', 'asascore', 'BMI'])

# ── Stage 2: + Comorbidities & Lifestyle ─────────────────────────────────────
COMORBIDITIES = get_existing(['CCI', 'comaf', 'comcarritme'])
LIFESTYLE     = get_existing(['roken', 'roken_missing', 'gewverl', 'dietist', 'anamok'])
STAGE2        = STAGE1 + COMORBIDITIES + LIFESTYLE

# ── Stage 3: + Tumor & Neoadjuvant ───────────────────────────────────────────
TUMOR  = get_existing(['scorect', 'scorecn']) + get_ohe_cols('tumlok1') + \
         get_ohe_cols('tumhist') + get_ohe_cols('tumkeus')

_neoadjsch_miss = df_encoded['neoadjsch'].isna().mean() * 100 if 'neoadjsch' in df_encoded.columns else 100
_neoadjsch_vars = ['neoadjsch'] if _neoadjsch_miss < 50 else []
NEOADJ = get_existing(['neoadjaf'] + _neoadjsch_vars) + get_ohe_cols('neoadjtype')
STAGE3 = STAGE2 + TUMOR + NEOADJ

# ── Stage 4: + Surgical Factors ──────────────────────────────────────────────
# bloedverlies_missing and duur_operatie_missing are included alongside the
# imputed values so the model can learn that "not recorded" is itself informative.
SURGICAL = get_existing([
    'conversie', 'duur_operatie', 'duur_operatie_missing',
    'bloedverlies', 'bloedverlies_missing',
    'resadd', 'curaok', 'salvok', 'emr'
]) + get_ohe_cols('typok') + get_ohe_cols('aardok') + \
   get_ohe_cols('procok') + get_ohe_cols('recontype') + get_ohe_cols('analig')
STAGE4 = STAGE3 + SURGICAL

# ── Stage 5: + Pulmonary function (fev1vc) — sensitivity analysis ─────────────
PULM   = get_existing(['fev1vc'])
STAGE5 = STAGE4 + PULM

def dedup(lst): return list(dict.fromkeys(lst))
STAGE1, STAGE2, STAGE3, STAGE4, STAGE5 = map(dedup, [STAGE1, STAGE2, STAGE3, STAGE4, STAGE5])

stages = {
    'Stage 1: Demographics + Fitness':   STAGE1,
    'Stage 2: + CCI + Lifestyle':         STAGE2,
    'Stage 3: + Tumor + Neoadjuvant':     STAGE3,
    'Stage 4: + Surgical Factors':        STAGE4,
}

print("Predictor counts per stage:")
for name, cols in stages.items():
    print(f"  {name:<45} {len(cols):>3} predictors")
print(f"  {'Stage 5: + fev1vc (subset)':<45} {len(STAGE5):>3} predictors")
print(f"\n  CCI covers: {list(CCI_WEIGHTS.keys())}")
print(f"  Kept separate: comaf, comcarritme (not in CCI, high clinical relevance)")
print(f"  Missing indicators added: bloedverlies_missing, duur_operatie_missing")

In [ ]:
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import Ridge

def prepare_xy(feature_cols, outcome_col='compl', data=None):
    """Build X, y for a given stage/outcome. Missing values are LEFT AS NaN here —
    imputation is done inside the per-fold Pipeline in run_cv/run_cv_regression
    (and friends) so the imputer is fit on training folds only, never on the
    full dataset. This avoids leaking validation-fold information into the
    imputation statistics (mean/median/mode)."""
    if data is None:
        data = df_encoded
    cols_present = [c for c in feature_cols if c in data.columns]
    sub = data[cols_present + [outcome_col]].copy()
    sub[outcome_col] = pd.to_numeric(sub[outcome_col], errors='coerce')
    sub = sub.dropna(subset=[outcome_col])

    X = sub[cols_present].copy()
    y = sub[outcome_col].values

    obj_cols = [c for c in X.columns if X[c].dtype == object]
    if obj_cols:
        print(f"  [prepare_xy] WARNING: dropping {len(obj_cols)} non-numeric column(s): {obj_cols}")
        X = X.drop(columns=obj_cols)

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    # NOTE: no fillna here — NaNs are imputed per-fold inside the modeling Pipeline.
    return X, y.astype(int) if y.dtype != float else y


def build_preprocessor(X):
    """Column-wise imputer: most-frequent for binary (<=2 unique values) columns,
    median for everything else — matches the strategy previously used in
    prepare_xy, but now fit fresh on each training fold instead of once globally.
    Column *type* (binary vs continuous) is structural metadata, not a fitted
    statistic, so determining it from the full X here does not leak fold info;
    the imputer itself (which DOES fit a statistic) is fit inside the Pipeline
    per training fold only."""
    binary_cols = [c for c in X.columns if X[c].nunique(dropna=True) <= 2]
    continuous_cols = [c for c in X.columns if c not in binary_cols]

    transformers = []
    if binary_cols:
        transformers.append(('bin_impute', SimpleImputer(strategy='most_frequent'), binary_cols))
    if continuous_cols:
        transformers.append(('cont_impute', SimpleImputer(strategy='median'), continuous_cols))

    return ColumnTransformer(transformers, remainder='drop')


def make_model_pipeline(X, model_name='LR', y=None):
    """Build a fresh Pipeline([impute, (scale,) model]) for the given model_name.
    Imputation (and scaling, for linear models) happens inside the pipeline so it
    is fit per-CV-fold on training data only — not once on the full dataset."""
    preprocessor = build_preprocessor(X)

    if model_name == 'LR':
        # LASSO with CV over C — used for primary staged models
        clf = LogisticRegressionCV(
            Cs=[0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
            penalty='l1', solver='liblinear', cv=3,
            class_weight='balanced', max_iter=2000,
            random_state=RANDOM_STATE
        )
        return Pipeline([
            ('impute', preprocessor),
            ('scaler', StandardScaler()),
            ('clf', clf)
        ])
    elif model_name == 'LR_fixed':
        # Fixed C=1.0 LR — used for secondary outcomes where LASSO over-penalizes
        clf = LogisticRegression(
            C=1.0, penalty='l1', solver='liblinear',
            class_weight='balanced', max_iter=2000,
            random_state=RANDOM_STATE
        )
        return Pipeline([
            ('impute', preprocessor),
            ('scaler', StandardScaler()),
            ('clf', clf)
        ])
    elif model_name == 'RF':
        clf = RandomForestClassifier(
            n_estimators=300, max_depth=6, min_samples_leaf=10,
            class_weight='balanced',
            random_state=RANDOM_STATE, n_jobs=-1
        )
        return Pipeline([
            ('impute', preprocessor),
            ('clf', clf)
        ])
    elif model_name == 'Ridge':
        reg = Ridge(alpha=1.0)
        return Pipeline([
            ('impute', preprocessor),
            ('scaler', StandardScaler()),
            ('reg', reg)
        ])
    elif model_name == 'RF_reg':
        reg = RandomForestRegressor(
            n_estimators=300, max_depth=6, min_samples_leaf=10,
            random_state=RANDOM_STATE, n_jobs=-1
        )
        return Pipeline([
            ('impute', preprocessor),
            ('reg', reg)
        ])
    else:  # XGB (classification or regression, decided by caller's estimator)
        if y is not None:
            scale_pos = (y == 0).sum() / max((y == 1).sum(), 1)
            clf = XGBClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                scale_pos_weight=scale_pos,
                use_label_encoder=False, eval_metric='logloss',
                random_state=RANDOM_STATE, verbosity=0
            )
        else:
            from xgboost import XGBRegressor
            clf = XGBRegressor(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbosity=0
            )
        return Pipeline([
            ('impute', preprocessor),
            ('clf', clf)
        ])


def run_cv(X, y, model_name='LR'):
    """5-fold stratified CV for binary classification. Returns dict of metrics.
    Imputation (+ scaling, for linear models) is fit fresh inside the Pipeline
    on each training fold only — see make_model_pipeline / build_preprocessor."""
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    aucs, baccs, briers, sens_list, spec_list = [], [], [], [], []

    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        model = make_model_pipeline(X_tr, model_name, y=y_tr)
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        pred  = model.predict(X_te)
        aucs.append(roc_auc_score(y_te, proba))
        baccs.append(balanced_accuracy_score(y_te, pred))
        briers.append(brier_score_loss(y_te, proba))
        tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
        sens_list.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        spec_list.append(tn / (tn + fp) if (tn + fp) > 0 else 0)

    return {
        'AUC':     np.mean(aucs),   'AUC_std':  np.std(aucs),
        'BalAcc':  np.mean(baccs),  'Brier':    np.mean(briers),
        'Sens':    np.mean(sens_list), 'Spec':  np.mean(spec_list),
    }


def run_cv_regression(X, y, model_name='RF'):
    """5-fold CV for continuous outcomes (LOS). Returns R², RMSE, MAE.
    Imputation (+ scaling, for Ridge) is fit fresh inside the Pipeline on each
    training fold only."""
    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

    cv = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    pipeline_name = 'RF_reg' if model_name == 'RF' else model_name

    r2s, rmses, maes = [], [], []
    for train_idx, test_idx in cv.split(X):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        model = make_model_pipeline(X_tr, pipeline_name)
        model.fit(X_tr, y_tr)
        pred = model.predict(X_te)
        r2s.append(r2_score(y_te, pred))
        rmses.append(np.sqrt(mean_squared_error(y_te, pred)))
        maes.append(mean_absolute_error(y_te, pred))

    return {
        'R2':   np.mean(r2s),   'R2_std':   np.std(r2s),
        'RMSE': np.mean(rmses), 'MAE':      np.mean(maes),
    }


print("Helper functions defined — LR (LASSO-CV), LR_fixed (C=1), RF, XGBoost.")
print("Imputation now happens per-CV-fold inside a Pipeline (no global pre-imputation).")

---
## 5. Model Training Helper Functions

In [ ]:
results = []

for stage_name, feat_cols in stages.items():
    X, y = prepare_xy(feat_cols, outcome_col='compl')
    print(f"\n{'='*65}")
    print(f"{stage_name}")
    print(f"  n={len(y)}, events={y.sum()} ({y.mean()*100:.1f}%), features={X.shape[1]}")

    for model_name, label in [('LR', 'LASSO LR'), ('RF', 'Random Forest'), ('XGB', 'XGBoost')]:
        if model_name == 'XGB' and not XGB_AVAILABLE:
            continue
        res = run_cv(X, y, model_name)
        print(f"  {label:<15} AUC={res['AUC']:.3f} ± {res['AUC_std']:.3f}  "
              f"BalAcc={res['BalAcc']:.3f}  Sens={res['Sens']:.3f}  Spec={res['Spec']:.3f}")
        results.append({'Stage': stage_name, 'Model': label, **res})

results_df = pd.DataFrame(results)
print("\n\nAll stages complete.")

---
## 8. Results Summary Table

---
## 9. AUC Progression Plot

How much does each added predictor group improve discrimination?

In [ ]:
if 'fev1vc' in df_encoded.columns:
    df_pulm = df_encoded.dropna(subset=['fev1vc'])
    n_pulm  = len(df_pulm)
    print(f"Patients with fev1vc data: {n_pulm} ({n_pulm/len(df_encoded)*100:.1f}% of total)")

    sens_results = []
    model_pairs = [('LR', 'LASSO LR'), ('RF', 'Random Forest'), ('XGB', 'XGBoost')]

    # Stage 4 on subset (baseline for comparison)
    X4s, y4s = prepare_xy(STAGE4, outcome_col='compl', data=df_pulm)
    print(f"  Stage 4 on subset: n={len(y4s)}, events={y4s.sum()}")
    s4_aucs = {}
    for mn, label in model_pairs:
        if mn == 'XGB' and not XGB_AVAILABLE:
            continue
        r = run_cv(X4s, y4s, mn)
        s4_aucs[mn] = r['AUC']
        print(f"  {label:<15} AUC={r['AUC']:.3f}")
        sens_results.append({'Stage': 'Stage 4 (fev1vc subset)', 'Model': label, **r})

    # Stage 5 on subset (with fev1vc added)
    X5s, y5s = prepare_xy(STAGE5, outcome_col='compl', data=df_pulm)
    print(f"\n  Stage 5 on subset: n={len(y5s)}, features={X5s.shape[1]}")
    for mn, label in model_pairs:
        if mn == 'XGB' and not XGB_AVAILABLE:
            continue
        r = run_cv(X5s, y5s, mn)
        delta = r['AUC'] - s4_aucs.get(mn, 0)
        print(f"  {label:<15} AUC={r['AUC']:.3f}  (Δ from Stage 4 subset: {delta:+.3f})")
        sens_results.append({'Stage': 'Stage 5: + fev1vc', 'Model': label, **r})

    sens_df = pd.DataFrame(sens_results)
else:
    print("fev1vc column not found — Stage 5 skipped.")
    sens_df = pd.DataFrame()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

short_labels = {
    'Stage 1: Demographics + Fitness':  'S1\nDemo+Fit',
    'Stage 2: + CCI + Lifestyle':        'S2\n+Comorbid',
    'Stage 3: + Tumor + Neoadjuvant':    'S3\n+Tumor',
    'Stage 4: + Surgical Factors':       'S4\n+Surgical',
}

model_colors = {
    'LASSO LR':      'steelblue',
    'Random Forest': 'seagreen',
    'XGBoost':       'coral',
}

for ax, (model_name, color) in zip(axes, model_colors.items()):
    sub = results_df[results_df['Model'] == model_name]
    if sub.empty:
        ax.set_title(f'{model_name} — no data', fontsize=12)
        continue

    xlabels = [short_labels.get(s, s) for s in sub['Stage']]
    x_pos   = range(len(xlabels))

    ax.plot(x_pos, sub['AUC'], marker='o', color=color, linewidth=2.5,
            markersize=9, zorder=3)
    ax.fill_between(
        x_pos,
        sub['AUC'] - sub['AUC_std'],
        sub['AUC'] + sub['AUC_std'],
        alpha=0.2, color=color, label='±1 SD'
    )

    for xi, (_, row) in zip(x_pos, sub.iterrows()):
        ax.annotate(f"{row['AUC']:.3f}",
                    xy=(xi, row['AUC']),
                    xytext=(0, 12), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold', color=color)

    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Random (0.5)')
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels(xlabels, fontsize=10)
    ax.set_ylim(0.45, 0.85)
    ax.set_ylabel('AUC-ROC (5-fold CV)', fontsize=12)
    ax.set_title(f'{model_name} — AUC across stages', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Prediction of Any Complication — AUC by Stage', fontweight='bold', fontsize=15)
plt.tight_layout()
plt.savefig('fig_auc_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
metrics       = ['AUC', 'BalAcc', 'Sens', 'Spec']
metric_labels = ['AUC-ROC', 'Balanced Acc.', 'Sensitivity', 'Specificity']
models_order  = ['LASSO LR', 'Random Forest', 'XGBoost']
colors        = ['steelblue', 'seagreen', 'coral']

stage4_sub = results_df[results_df['Stage'] == 'Stage 4: + Surgical Factors']

x     = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
for i, (model, color) in enumerate(zip(models_order, colors)):
    row = stage4_sub[stage4_sub['Model'] == model]
    if row.empty:
        continue
    vals = [row[m].values[0] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=model,
                  color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                f'{val:.3f}', ha='center', fontsize=9)

ax.set_xticks(x + width)
ax.set_xticklabels(metric_labels, fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score (5-fold CV)', fontsize=12)
ax.set_title('Stage 4 Model Performance — Any Complication', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import StratifiedKFold

def get_oof_probas(X, y, model_name='XGB'):
    """Collect out-of-fold predicted probabilities (for a single pooled ROC).
    Uses make_model_pipeline so imputation (+ scaling, for LR) is fit fresh on
    each training fold only — consistent with run_cv."""
    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(y))

    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr = y[train_idx]
        model = make_model_pipeline(X_tr, model_name, y=y_tr)
        model.fit(X_tr, y_tr)
        oof_proba[test_idx] = model.predict_proba(X_te)[:, 1]

    return oof_proba


# ── Plot 1: Stage progression per model (3 subplots) ─────────────────────────
stage_palette = plt.cm.Blues(np.linspace(0.35, 0.9, len(stages)))
model_configs  = [('LR', 'LASSO LR', 'steelblue'),
                  ('RF', 'Random Forest', 'seagreen'),
                  ('XGB', 'XGBoost', 'coral')]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (mn, label, color) in zip(axes, model_configs):
    if mn == 'XGB' and not XGB_AVAILABLE:
        ax.set_title(f'{label} — not available')
        continue

    stage_pal = plt.cm.Blues(np.linspace(0.35, 0.9, len(stages)))
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')

    for i, (stage_name, feat_cols) in enumerate(stages.items()):
        X, y = prepare_xy(feat_cols, outcome_col='compl')
        oof  = get_oof_probas(X, y, mn)
        fpr, tpr, _ = roc_curve(y, oof)
        auc_val = roc_auc_score(y, oof)
        short = stage_name.split(':')[0]
        ax.plot(fpr, tpr, color=stage_pal[i], linewidth=2.2,
                label=f"{short} (AUC={auc_val:.3f})")

    ax.set_xlabel('1 - Specificity', fontsize=11)
    ax.set_ylabel('Sensitivity', fontsize=11)
    ax.set_title(f'ROC — {label}', fontweight='bold', fontsize=12)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.2)

plt.suptitle('ROC Curves by Stage — All Models\n(Primary outcome: any complication)',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

# ── Plot 2: Stage 4 model comparison on one plot ──────────────────────────────
X4, y4 = prepare_xy(STAGE4, outcome_col='compl')

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')

for mn, label, color in model_configs:
    if mn == 'XGB' and not XGB_AVAILABLE:
        continue
    oof = get_oof_probas(X4, y4, mn)
    fpr, tpr, _ = roc_curve(y4, oof)
    auc_val = roc_auc_score(y4, oof)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f"{label} (AUC={auc_val:.3f})")

ax.set_xlabel('1 - Specificity', fontsize=12)
ax.set_ylabel('Sensitivity', fontsize=12)
ax.set_title('Stage 4 — Model Comparison\n(Any complication)',
             fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Calibration Analysis — Stage 4 ─────────────────────────────────────
import os

print('Computing OOF probabilities for calibration analysis...')
X4_cal, y4_cal = prepare_xy(STAGE4, outcome_col='compl')

oof_lr  = get_oof_probas(X4_cal, y4_cal, 'LR')
oof_rf  = get_oof_probas(X4_cal, y4_cal, 'RF')
oof_xgb = get_oof_probas(X4_cal, y4_cal, 'XGB') if XGB_AVAILABLE else None

model_probas = {'LASSO LR': oof_lr, 'Random Forest': oof_rf}
if oof_xgb is not None:
    model_probas['XGBoost'] = oof_xgb

colors_cal = {'LASSO LR': 'steelblue', 'Random Forest': 'seagreen', 'XGBoost': 'tomato'}
null_brier = brier_score_loss(y4_cal, [y4_cal.mean()] * len(y4_cal))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Perfect calibration')

print(f'Brier Scores — Stage 4  (null model: {null_brier:.4f})')
for name, probs in model_probas.items():
    prob_true, prob_pred = calibration_curve(y4_cal, probs, n_bins=10, strategy='quantile')
    bs = brier_score_loss(y4_cal, probs)
    ax.plot(prob_pred, prob_true, 'o-', color=colors_cal[name],
            linewidth=2, markersize=6, label=f'{name}  (Brier = {bs:.3f})')
    print(f'  {name:<18}: {bs:.4f}')

ax.set_xlabel('Mean predicted probability', fontsize=12)
ax.set_ylabel('Observed fraction of positives', fontsize=12)
ax.set_title('Calibration Plot — Stage 4\n(Primary outcome: any complication)', fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

ax2 = axes[1]
for name, probs in model_probas.items():
    ax2.hist(probs, bins=30, alpha=0.55, label=name,
             color=colors_cal[name], edgecolor='white')
ax2.axvline(y4_cal.mean(), color='black', linestyle='--', linewidth=1.5,
            label=f'True prevalence ({y4_cal.mean():.2f})')
ax2.set_xlabel('Predicted probability of complication', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Distribution of Predicted Probabilities\n(Stage 4)', fontweight='bold')
ax2.legend(fontsize=10)

plt.tight_layout()
# Source data isn't included in this repo — figures are saved to a local ../figures dir instead.
save_dir = "../figures"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'fig_calibration.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_calibration.png')


In [ ]:
if SHAP_AVAILABLE:
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier

    X4, y4 = prepare_xy(STAGE4, outcome_col='compl')
    X_train, X_test, y_train, y_test = train_test_split(
        X4, y4, test_size=0.25, stratify=y4, random_state=RANDOM_STATE
    )

    # Random Forest — best CV model at Stage 4 (AUC 0.615)
    # Impute on the train split only (median for continuous, most-frequent for
    # binary), then fit the RF — mirrors the per-fold imputation used in CV.
    shap_preprocessor = build_preprocessor(X_train)
    X_train_imp = pd.DataFrame(
        shap_preprocessor.fit_transform(X_train),
        columns=shap_preprocessor.get_feature_names_out(),
        index=X_train.index
    )
    X_test_imp = pd.DataFrame(
        shap_preprocessor.transform(X_test),
        columns=shap_preprocessor.get_feature_names_out(),
        index=X_test.index
    )
    # Strip the ColumnTransformer's prefix (e.g. "bin_impute__", "cont_impute__")
    # so feature names match the original columns for SHAP labeling below.
    rename_map = {c: c.split('__', 1)[1] for c in X_train_imp.columns}
    X_train_imp = X_train_imp.rename(columns=rename_map)
    X_test_imp  = X_test_imp.rename(columns=rename_map)

    rf_shap = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=10,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_shap.fit(X_train_imp, y_train)

    explainer   = shap.TreeExplainer(rf_shap)
    shap_values = explainer.shap_values(X_test_imp)
    # RF may return a list [class_0, class_1] or a single 2D array depending on SHAP version
    if isinstance(shap_values, list):
        shap_vals_pos = shap_values[1]  # list format: use positive class (complication = yes)

    elif hasattr(shap_values, "ndim") and shap_values.ndim == 3:
        shap_vals_pos = shap_values[:, :, 1]
    else:
        shap_vals_pos = shap_values     # already 2D array for binary classification

    # Dutch → English label mapping
    label_map = {
        'age_at_op':             'Age at surgery',
        'geslacht':              'Sex (male)',
        'BMI':                   'BMI',
        'asascore':              'ASA score',
        'gewverl':               'Preop. weight loss',
        'CCI':                   'Charlson Comorbidity Index',
        'roken':                 'Smoking status',
        'roken_missing':         'Smoking: unknown',
        'comaf':                 'Atrial fibrillation',
        'comcarritme':           'Cardiac arrhythmia',
        'dietist':               'Dietitian referral',
        'anamok':                'Prior abdominal surgery',
        'scorect':               'Clinical T stage',
        'scorecn':               'Clinical N stage',
        'neoadjaf':              'Neoadjuvant response',
        'neoadjsch':             'Neoadjuvant: cycles completed',
        'tumhist_1.0':           'Histology: adenocarcinoma',
        'tumhist_2.0':           'Histology: squamous cell',
        'tumhist_7.0':           'Histology: other',
        'tumkeus_1.0':           'Tumor: resectable',
        'tumkeus_2.0':           'Tumor: borderline',
        'tumlok1_3.0':           'Tumor location: mid-oesophagus',
        'tumlok1_4.0':           'Tumor location: distal oesophagus',
        'tumlok1_5.0':           'Tumor location: GEJ',
        'neoadjtype_1.0':        'Neoadjuvant: chemotherapy',
        'neoadjtype_2.0':        'Neoadjuvant: chemoradiotherapy',
        'neoadjtype_3.0':        'Neoadjuvant: radiotherapy',
        'neoadjtype_4.0':        'Neoadjuvant: other',
        'duur_operatie':         'Operation duration (log)',
        'duur_operatie_missing': 'Op. duration: not recorded',
        'bloedverlies':          'Intraop. blood loss (log)',
        'bloedverlies_missing':  'Blood loss: not recorded',
        'conversie':             'Conversion to open',
        'curaok':                'Curative intent',
        'salvok':                'Salvage surgery',
        'emr':                   'Prior EMR',
        'resadd':                'Additional resection',
        'procok_1.0':            'Approach: open thoracic',
        'procok_2.0':            'Approach: hybrid',
        'procok_3.0':            'Approach: robotic',
        'procok_4.0':            'Approach: MIE',
        'procok_1':              'Approach: open thoracic',
        'procok_2':              'Approach: hybrid',
        'procok_3':              'Approach: robotic',
        'procok_4':              'Approach: MIE',
        'analig_1.0':            'Anastomosis: cervical',
        'analig_2.0':            'Anastomosis: intrathoracic',
        'analig_3.0':            'Anastomosis: abdominal',
        'recontype_1.0':         'Reconstruction: gastric tube',
        'recontype_4.0':         'Reconstruction: colon',
        'aardok_1.0':            'Procedure: resection',
        'typok_0.0':             'Operation type: elective',
        'reslmm_1.0':            'Resection margin: R0',
        'fev1vc':                'FEV1/VC (%)',
    }

    X_test_en = X_test_imp.rename(columns=label_map)

    # ── Beeswarm (dot) plot ─────────────────────────────────────────────
    plt.figure(figsize=(12, 9))
    shap.summary_plot(shap_vals_pos, X_test_en, plot_type='dot',
                      max_display=20, show=False)
    plt.title('SHAP Summary — Stage 4 Random Forest\n(Primary outcome: any complication)',
              fontweight='bold', fontsize=14)
    plt.xlabel('SHAP value (impact on model output)', fontsize=12)
    plt.tick_params(axis='both', labelsize=11)
    plt.tight_layout()
    plt.savefig('shap_beeswarm.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: shap_beeswarm.png")

    # ── Bar plot: mean |SHAP| ────────────────────────────────────────────
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_vals_pos, X_test_en, plot_type='bar',
                      max_display=20, show=False)
    plt.title('Mean |SHAP| — Top 20 Predictors (Stage 4 Random Forest)',
              fontweight='bold', fontsize=14)
    plt.xlabel('mean(|SHAP value|) (average impact on model output)', fontsize=12)
    plt.tick_params(axis='both', labelsize=11)
    plt.tight_layout()
    plt.savefig('shap_bar.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Saved: shap_bar.png")

else:
    print("Install shap: pip install shap")


In [ ]:
# ── Hyperparameter Sensitivity — Random Forest, Stage 4 ─────────────────
import os

print('RF hyperparameter sensitivity — Stage 4 (any complication)\n')
X4_hs, y4_hs = prepare_xy(STAGE4, outcome_col='compl')

depth_vals    = [4, 6, 8, None]
min_leaf_vals = [5, 10, 20]
hs_rows       = []

for depth in depth_vals:
    for min_leaf in min_leaf_vals:
        cv_hs = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        fold_aucs = []
        for tr, te in cv_hs.split(X4_hs, y4_hs):
            X_tr_hs, X_te_hs = X4_hs.iloc[tr], X4_hs.iloc[te]
            y_tr_hs, y_te_hs = y4_hs[tr], y4_hs[te]
            # Impute per-fold (fit on train only) before fitting RF, consistent with run_cv.
            preproc_hs = build_preprocessor(X_tr_hs)
            rf_tmp = RandomForestClassifier(
                n_estimators=300, max_depth=depth, min_samples_leaf=min_leaf,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
            )
            pipe_hs = Pipeline([('impute', preproc_hs), ('clf', rf_tmp)])
            pipe_hs.fit(X_tr_hs, y_tr_hs)
            fold_aucs.append(roc_auc_score(
                y_te_hs, pipe_hs.predict_proba(X_te_hs)[:, 1]))
        dlabel = str(depth) if depth is not None else 'None'
        hs_rows.append({'max_depth': dlabel, 'min_samples_leaf': min_leaf,
                        'AUC_mean': round(float(np.mean(fold_aucs)), 4),
                        'AUC_std':  round(float(np.std(fold_aucs)),  4)})
        print(f'  max_depth={dlabel:<5}  min_samples_leaf={min_leaf}  '
              f'AUC={np.mean(fold_aucs):.3f} +/- {np.std(fold_aucs):.3f}')

hs_df = pd.DataFrame(hs_rows)
pivot = hs_df.pivot(index='max_depth', columns='min_samples_leaf', values='AUC_mean')

import seaborn as _sns
fig, ax = plt.subplots(figsize=(7, 5))
_sns.heatmap(pivot, annot=True, fmt='.3f', cmap='Blues', ax=ax,
             vmin=pivot.values.min() - 0.005,
             vmax=pivot.values.max() + 0.005,
             linewidths=0.5, cbar_kws={'label': 'AUC-ROC (5-fold CV)'})
ax.set_title('RF Hyperparameter Sensitivity — Stage 4 AUC\n(5-fold CV, any complication)',
             fontweight='bold')
ax.set_xlabel('min_samples_leaf', fontsize=12)
ax.set_ylabel('max_depth', fontsize=12)

depth_order = [str(d) if d is not None else 'None' for d in depth_vals]
try:
    r = depth_order.index('6')
    c = list(min_leaf_vals).index(10)
    ax.add_patch(plt.Rectangle((c, r), 1, 1, fill=False, edgecolor='red', lw=3))
    ax.text(c + 0.5, r - 0.12, 'chosen', ha='center', va='bottom',
            color='red', fontsize=9, fontweight='bold')
except ValueError:
    pass

plt.tight_layout()
# Source data isn't included in this repo — figures are saved to a local ../figures dir instead.
save_dir = "../figures"
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, 'fig_hyperparam_sensitivity.png'), dpi=150, bbox_inches='tight')
plt.show()

best = hs_df.loc[hs_df['AUC_mean'].idxmax()]
chosen_row = hs_df[(hs_df['max_depth'] == '6') & (hs_df['min_samples_leaf'] == 10)]
if not chosen_row.empty:
    chosen_auc = chosen_row['AUC_mean'].values[0]
    print(f'Best config:    max_depth={best["max_depth"]}, '
          f'min_leaf={best["min_samples_leaf"]},  AUC={best["AUC_mean"]:.3f}')
    print(f'Chosen config:  max_depth=6,  min_leaf=10,  AUC={chosen_auc:.3f}')
    print(f'Difference:     {best["AUC_mean"] - chosen_auc:+.4f}')
print('Conclusion: chosen hyperparameters are at or within 0.005 AUC of the grid optimum.')


---
## 13. Secondary Outcomes — Stage 4 Model

We now apply the same Stage 4 predictor set to predict specific complication types.
Only outcomes with ≥100 events are modeled.

In [ ]:
secondary_outcomes = {
    'comppulm':   'Pulmonary complication',
    'compcar':    'Cardiac complication',
    'compand':    'Anastomotic leak',
    'compinfect': 'Infection',
}

sec_results = []

for col, label in secondary_outcomes.items():
    if col not in df_encoded.columns:
        print(f"  {label}: column not found")
        continue

    X_sec, y_sec = prepare_xy(STAGE4, outcome_col=col)
    n_events = y_sec.sum()

    if n_events < 50:
        print(f"  {label}: only {n_events} events — too few to model")
        continue

    print(f"\n{label}  (n={len(y_sec)}, events={n_events}, {n_events/len(y_sec)*100:.1f}%)")

    # Use LR_fixed (C=1.0) instead of LASSO-CV for secondary outcomes —
    # LASSO over-penalizes when signal is weak, collapsing all coefficients to 0
    lr_s = run_cv(X_sec, y_sec, 'LR_fixed')
    print(f"  LR  AUC={lr_s['AUC']:.3f} \u00b1 {lr_s['AUC_std']:.3f}")
    sec_results.append({'Outcome': label, 'Model': 'LR (C=1)', **lr_s})

    rf_s = run_cv(X_sec, y_sec, 'RF')
    print(f"  RF  AUC={rf_s['AUC']:.3f} \u00b1 {rf_s['AUC_std']:.3f}")
    sec_results.append({'Outcome': label, 'Model': 'Random Forest', **rf_s})

    if XGB_AVAILABLE:
        xgb_s = run_cv(X_sec, y_sec, 'XGB')
        print(f"  XGB AUC={xgb_s['AUC']:.3f} \u00b1 {xgb_s['AUC_std']:.3f}")
        sec_results.append({'Outcome': label, 'Model': 'XGBoost', **xgb_s})

sec_df = pd.DataFrame(sec_results)

In [ ]:
# Secondary outcomes comparison chart
if not sec_df.empty:
    fig, ax = plt.subplots(figsize=(12, 5))

    outcomes_uniq = sec_df['Outcome'].unique()
    models_uniq   = sec_df['Model'].unique()
    n_models = len(models_uniq)
    x = np.arange(len(outcomes_uniq))
    width = 0.25
    palette = ['steelblue', 'mediumseagreen', 'coral']

    for i, (model, color) in enumerate(zip(models_uniq, palette)):
        sub = sec_df[sec_df['Model'] == model]
        aucs = [sub[sub['Outcome'] == o]['AUC'].values[0]
                if len(sub[sub['Outcome'] == o]) > 0 else 0
                for o in outcomes_uniq]
        stds = [sub[sub['Outcome'] == o]['AUC_std'].values[0]
                if len(sub[sub['Outcome'] == o]) > 0 else 0
                for o in outcomes_uniq]
        offset = (i - (n_models - 1) / 2) * width
        bars = ax.bar(x + offset, aucs, width, yerr=stds, capsize=4,
                      label=model, color=color, alpha=0.8, edgecolor='white')
        for bar, auc in zip(bars, aucs):
            if auc > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
                        f'{auc:.3f}', ha='center', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(outcomes_uniq, fontsize=11)
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
    ax.set_ylabel('AUC-ROC (5-fold CV)', fontsize=12)
    ax.set_title('Secondary Outcomes \u2014 Stage 4 Predictor Set', fontweight='bold', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 15.3  Complication type multiclass ───────────────────────────────────────
# 0=None  1=Pulmonary  2=Anastomotic leak  3=Cardiac  4=Other
from sklearn.metrics import f1_score

print("Complication Type — Multiclass Classification")
print("=" * 55)

multiclass_results = []
if 'comp_type' in df_encoded.columns:
    X_mc, y_mc = prepare_xy(STAGE4, outcome_col='comp_type')
    print(f"n={len(y_mc)}  Class distribution:")
    labels_mc = {0:'None', 1:'Pulmonary', 2:'Leakage', 3:'Cardiac', 4:'Other'}
    for k, v in sorted(pd.Series(y_mc).value_counts().items()):
        print(f"  {labels_mc.get(k, k):<15} {v:>4} ({v/len(y_mc)*100:.1f}%)")

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    for mn in ['RF', 'XGB']:
        if mn == 'XGB' and not XGB_AVAILABLE:
            continue

        f1s, baccs = [], []
        for train_idx, test_idx in cv.split(X_mc, y_mc):
            X_tr_mc, X_te_mc = X_mc.iloc[train_idx], X_mc.iloc[test_idx]
            y_tr_mc, y_te_mc = y_mc[train_idx], y_mc[test_idx]
            # Impute per-fold (fit on train only), consistent with run_cv.
            preproc_mc = build_preprocessor(X_tr_mc)
            if mn == 'RF':
                model = RandomForestClassifier(
                    n_estimators=300, max_depth=6, min_samples_leaf=10,
                    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
                )
            else:
                model = XGBClassifier(
                    n_estimators=300, learning_rate=0.05, max_depth=4,
                    subsample=0.8, colsample_bytree=0.8,
                    use_label_encoder=False, eval_metric='mlogloss',
                    random_state=RANDOM_STATE, verbosity=0
                )
            pipe_mc = Pipeline([('impute', preproc_mc), ('clf', model)])
            pipe_mc.fit(X_tr_mc, y_tr_mc)
            pred = pipe_mc.predict(X_te_mc)
            f1s.append(f1_score(y_te_mc, pred, average='macro', zero_division=0))
            baccs.append(balanced_accuracy_score(y_te_mc, pred))

        print(f"\n  {mn}  Macro-F1={np.mean(f1s):.3f} ± {np.std(f1s):.3f}  "
              f"BalAcc={np.mean(baccs):.3f}")
        multiclass_results.append({'Model': mn, 'MacroF1': np.mean(f1s),
                                   'MacroF1_std': np.std(f1s), 'BalAcc': np.mean(baccs)})

In [ ]:
# ── 15.2  Hospital LOS & ICU stay regression ─────────────────────────────────
# opnameduur = total hospital length of stay (days)
# icdg       = ICU-specific stay (days) — proxy for Clavien-Dindo grade 4

reg_results = []

for LOS_COL, label in [('opnameduur', 'Total hospital LOS'), ('icdg', 'ICU stay (CD grade 4 proxy)')]:
    if LOS_COL not in df_encoded.columns:
        print(f"  '{LOS_COL}' not found — skipping")
        continue

    X_los, y_los = prepare_xy(STAGE4, outcome_col=LOS_COL)
    y_los = y_los.astype(float)

    print(f"\n{label} ({LOS_COL})")
    print(f"  n={len(y_los)} | median={np.median(y_los):.0f}d | "
          f"mean={y_los.mean():.1f}d | range={y_los.min():.0f}–{y_los.max():.0f}d | "
          f"zeros={int((y_los==0).sum())} ({(y_los==0).mean()*100:.1f}%)")

    y_log = np.log1p(y_los)

    for mn in ['Ridge', 'RF', 'XGB']:
        if mn == 'XGB' and not XGB_AVAILABLE:
            continue
        r = run_cv_regression(X_los, y_log, mn)
        rmse_days = np.expm1(r['RMSE'])
        print(f"  {mn:<6} R²={r['R2']:.3f} ± {r['R2_std']:.3f}  "
              f"RMSE≈{rmse_days:.1f} days  MAE={r['MAE']:.3f}")
        reg_results.append({'Outcome': label, 'Model': mn, **r, 'RMSE_days': rmse_days})

reg_df = pd.DataFrame(reg_results)

# Distribution plots side by side
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row_i, (col, title) in enumerate([('opnameduur', 'Total hospital LOS'), ('icdg', 'ICU stay')]):
    if col not in df_encoded.columns:
        continue
    vals = pd.to_numeric(df_encoded[col], errors='coerce').dropna()
    axes[row_i][0].hist(vals, bins=40, color='steelblue', edgecolor='white')
    axes[row_i][0].axvline(vals.median(), color='red', linestyle='--',
                           label=f'Median: {vals.median():.0f}d')
    axes[row_i][0].set_title(f'{title} — raw', fontweight='bold')
    axes[row_i][0].set_xlabel('Days')
    axes[row_i][0].legend()

    axes[row_i][1].hist(np.log1p(vals), bins=40, color='coral', edgecolor='white')
    axes[row_i][1].set_title(f'{title} — log1p transformed', fontweight='bold')
    axes[row_i][1].set_xlabel('log1p(days)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 15.1  Binary outcomes ─────────────────────────────────────────────────────
binary_outcomes = {
    'compl':               'Any complication',
    'derived_clavien_bin': 'Severity: minor vs major (CD)',
    'heropn':              'Readmission',
    'icu_admission':       'ICU admission',
}

binary_results = []
for col, label in binary_outcomes.items():
    if col not in df_encoded.columns:
        print(f"  {label}: column not found — skipping")
        continue

    X_o, y_o = prepare_xy(STAGE4, outcome_col=col)
    n_events = y_o.sum()
    if n_events < 50:
        print(f"  {label}: only {n_events} events — too few to model reliably")
        continue

    print(f"\n{label}  (n={len(y_o)}, events={n_events}, {y_o.mean()*100:.1f}%)")
    # Use LR_fixed (C=1.0) instead of LASSO-CV — LASSO collapses when event rate is
    # low or signal is weak (Readmission ~15%, ICU ~28%, CD severity ~46%)
    for mn, label_model in [('LR_fixed', 'LR'), ('RF', 'RF'), ('XGB', 'XGB')]:
        if mn == 'XGB' and not XGB_AVAILABLE:
            continue
        r = run_cv(X_o, y_o, mn)
        print(f"  {label_model:<4} AUC={r['AUC']:.3f} ± {r['AUC_std']:.3f}  "
              f"BalAcc={r['BalAcc']:.3f}  Sens={r['Sens']:.3f}  Spec={r['Spec']:.3f}")
        binary_results.append({'Outcome': label, 'Model': label_model, **r})

binary_outcomes_df = pd.DataFrame(binary_results)

In [ ]:
# ── 15.4  Full outcomes summary table ─────────────────────────────────────────
print("\n" + "=" * 65)
print("ALL OUTCOMES SUMMARY — Stage 4 predictor set")
print("=" * 65)

print("\nBINARY OUTCOMES (AUC-ROC):")
if 'binary_outcomes_df' in dir() and not binary_outcomes_df.empty:
    print(binary_outcomes_df[['Outcome', 'Model', 'AUC', 'AUC_std', 'BalAcc', 'Sens', 'Spec']].to_string(index=False))
else:
    print("  (no binary outcomes computed)")

print("\nREGRESSION — Hospital LOS:")
if 'reg_df' in dir() and not reg_df.empty:
    print(reg_df[['Outcome', 'Model', 'R2', 'R2_std', 'RMSE_days', 'MAE']].to_string(index=False))
else:
    print("  (no regression results computed)")

print("\nMULTICLASS — Complication type (Macro-F1):")
if 'multiclass_results' in dir() and multiclass_results:
    print(pd.DataFrame(multiclass_results)[['Model', 'MacroF1', 'MacroF1_std', 'BalAcc']].to_string(index=False))
else:
    print("  (no multiclass results computed)")

In [ ]:
import os
# Source data isn't included in this repo — exported feature/outcome CSVs go to ./output instead.
EXPORT_DIR = "./output"
os.makedirs(EXPORT_DIR, exist_ok=True)

X_export_raw, _ = prepare_xy(STAGE4, outcome_col='compl')
# For this static export (not used in any CV loop), impute on the full Stage 4
# matrix so the saved CSV has no missing values — this matches what each CV
# fold's pipeline does internally, just applied once for the export artifact.
_export_preprocessor = build_preprocessor(X_export_raw)
X_export = pd.DataFrame(
    _export_preprocessor.fit_transform(X_export_raw),
    columns=_export_preprocessor.get_feature_names_out(),
    index=X_export_raw.index
)
X_export = X_export.rename(columns={c: c.split('__', 1)[1] for c in X_export.columns})
X_export = X_export[X_export_raw.columns]  # restore original column order

print("=" * 60)
print(f"STAGE 4 FEATURE MATRIX  —  X")
print("=" * 60)
print(f"Shape: {X_export.shape[0]} patients × {X_export.shape[1]} features\n")

demo_cols     = [c for c in STAGE1 if c in X_export.columns]
comorbid_cols = [c for c in STAGE2 if c not in STAGE1 and c in X_export.columns]
tumor_cols    = [c for c in STAGE3 if c not in STAGE2 and c in X_export.columns]
surgical_cols = [c for c in STAGE4 if c not in STAGE3 and c in X_export.columns]

print(f"  Stage 1 — Demographics ({len(demo_cols)}):    {demo_cols}")
print(f"  Stage 2 — Comorbidities/Lifestyle ({len(comorbid_cols)}): {comorbid_cols}")
print(f"  Stage 3 — Tumor/Neoadjuvant ({len(tumor_cols)}):  {tumor_cols}")
print(f"  Stage 4 — Surgical ({len(surgical_cols)}): {surgical_cols}")
print(f"\nMissing values after imputation: {X_export.isna().sum().sum()} (should be 0)")
print(f"\nFirst 5 rows of X:")
display(X_export.head())

print("\n" + "=" * 60)
print("OUTCOME VARIABLES  —  y")
print("=" * 60)

all_outcomes = {
    'compl':               'Any complication (primary)',
    'derived_clavien_bin': 'Clavien-Dindo binary (major vs minor)',
    'heropn':              'Readmission (binary)',
    'icu_admission':       'ICU admission (binary)',
    'opnameduur':          'Hospital LOS (days, regression)',
    'comp_type':           'Complication type (0-4, multiclass)',
}

y_df = pd.DataFrame({'upn': df_encoded['upn'] if 'upn' in df_encoded.columns else range(len(df_encoded))})
for col, label in all_outcomes.items():
    if col in df_encoded.columns:
        s = pd.to_numeric(df_encoded[col], errors='coerce')
        y_df[col] = s.values
        if col in ['compl', 'derived_clavien_bin', 'heropn', 'icu_admission']:
            n = int(s.sum())
            print(f"  {label:<40} n={len(s.dropna())}  events={n} ({n/s.notna().sum()*100:.1f}%)")
        elif col == 'opnameduur':
            print(f"  {label:<40} n={s.notna().sum()}  median={s.median():.0f}d  range={s.min():.0f}–{s.max():.0f}d")
        elif col == 'comp_type':
            dist = s.value_counts().sort_index()
            print(f"  {label:<40} classes: {dict(dist.items())}")

print(f"\nFirst 5 rows of outcomes:")
display(y_df.head())

X_save_path = os.path.join(EXPORT_DIR, "model_X_stage4.csv")
y_save_path = os.path.join(EXPORT_DIR, "model_y_outcomes.csv")
X_export.to_csv(X_save_path, index=False)
y_df.to_csv(y_save_path, index=False)
print(f"\nSaved X → {X_save_path}")
print(f"Saved y → {y_save_path}")

In [ ]:
print("=" * 70)
print("STAGED MODEL RESULTS — PRIMARY OUTCOME: any complication (compl)")
print("=" * 70)
print(results_df[['Stage', 'Model', 'AUC', 'AUC_std', 'BalAcc', 'Sens', 'Spec']].to_string(index=False))

if 'sens_df' in dir() and not sens_df.empty:
    print("\n" + "=" * 70)
    print("SENSITIVITY ANALYSIS — fev1vc subset")
    print("=" * 70)
    print(sens_df[['Stage', 'Model', 'AUC', 'AUC_std', 'BalAcc']].to_string(index=False))

if 'sec_df' in dir() and not sec_df.empty:
    print("\n" + "=" * 70)
    print("SECONDARY OUTCOMES — Stage 4 predictor set")
    print("=" * 70)
    print(sec_df[['Outcome', 'Model', 'AUC', 'AUC_std', 'BalAcc']].to_string(index=False))

# Best AUC across staged results
all_results = results_df.copy()
if 'sens_df' in dir() and not sens_df.empty:
    all_results = pd.concat([all_results, sens_df], ignore_index=True)

best = all_results.loc[all_results['AUC'].idxmax()]
print(f"\nBest result overall: {best['Stage']} — {best['Model']}")
print(f"  AUC = {best['AUC']:.3f} ± {best['AUC_std']:.3f}")
print(f"  Balanced Accuracy = {best['BalAcc']:.3f}")
print(f"  Sensitivity = {best['Sens']:.3f}")
print(f"  Specificity = {best['Spec']:.3f}")

---
## Combined AUC Progression — All Models

All three models plotted on a single panel to directly compare stage-by-stage information gain and identify which predictor group drives the largest improvement.

In [ ]:
# ── Combined Stage-by-Stage AUC Progression — all models on one panel ─────────
stage_keys = [
    'Stage 1: Demographics + Fitness',
    'Stage 2: + CCI + Lifestyle',
    'Stage 3: + Tumor + Neoadjuvant',
    'Stage 4: + Surgical Factors',
]
stage_labels = ['Stage 1\nDemo+Fit', 'Stage 2\n+Comorbid',
                'Stage 3\n+Tumor',  'Stage 4\n+Surgical']
model_colors = {'LASSO LR': 'steelblue', 'Random Forest': 'seagreen', 'XGBoost': 'coral'}

fig, ax = plt.subplots(figsize=(9, 5))

for model_name, color in model_colors.items():
    sub = results_df[results_df['Model'] == model_name]
    if sub.empty:
        continue
    aucs, stds = [], []
    for sk in stage_keys:
        row = sub[sub['Stage'] == sk]
        aucs.append(float(row['AUC'].values[0])     if len(row) > 0 else float('nan'))
        stds.append(float(row['AUC_std'].values[0]) if len(row) > 0 else 0.0)

    xp = list(range(4))
    ax.plot(xp, aucs, marker='o', color=color, linewidth=2.5, label=model_name)
    ax.fill_between(xp,
                    [a - s for a, s in zip(aucs, stds)],
                    [a + s for a, s in zip(aucs, stds)],
                    alpha=0.15, color=color)
    for xi, auc in zip(xp, aucs):
        if auc == auc:   # skip NaN
            ax.text(xi, auc + 0.008, f'{auc:.3f}',
                    ha='center', fontsize=9, color=color, fontweight='bold')

ax.set_xticks(range(4))
ax.set_xticklabels(stage_labels, fontsize=11)
ax.set_ylim(0.45, 0.72)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Chance (0.5)')
ax.set_ylabel('AUC-ROC (5-fold CV)', fontsize=12)
ax.set_title('Stage-by-Stage AUC Progression — All Models',
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print incremental AUC gains
print("\nIncremental AUC gain per stage transition:")
gain_labels = ['S1 -> S2  (+comorbidity)',
               'S2 -> S3  (+tumor/neoadj)',
               'S3 -> S4  (+surgical)   ']
for model_name in model_colors:
    sub = results_df[results_df['Model'] == model_name]
    if sub.empty:
        continue
    aucs = []
    for sk in stage_keys:
        row = sub[sub['Stage'] == sk]
        aucs.append(float(row['AUC'].values[0]) if len(row) > 0 else float('nan'))
    print(f"  {model_name}:")
    for j, lbl in enumerate(gain_labels):
        delta = aucs[j+1] - aucs[j]
        print(f"    {lbl}  delta = {delta:+.3f}")


---
## Error Analysis — Stage 4 Random Forest

We compare key clinical characteristics between the four prediction outcome groups (TP/FN/TN/FP) to understand which patients the model tends to miss or mis-flag.

In [ ]:
# ── Error Analysis — Stage 4 Random Forest ───────────────────────────────────
X4_ea, y4_ea = prepare_xy(STAGE4, outcome_col='compl')
oof_rf_ea    = get_oof_probas(X4_ea, y4_ea, 'RF')

threshold  = 0.5
y_pred_ea  = (oof_rf_ea >= threshold).astype(int)

masks = {
    'True Positive\n(hit)':          (y4_ea == 1) & (y_pred_ea == 1),
    'False Negative\n(missed)':      (y4_ea == 1) & (y_pred_ea == 0),
    'True Negative\n(clear)':        (y4_ea == 0) & (y_pred_ea == 0),
    'False Positive\n(false alarm)': (y4_ea == 0) & (y_pred_ea == 1),
}
print(f"Threshold = {threshold}")
for lbl, mask in masks.items():
    print(f"  {lbl.replace(chr(10),' ')}: n = {mask.sum()}")

df_ea = X4_ea.copy()
df_ea['group'] = 'Other'
for lbl, mask in masks.items():
    df_ea['group'] = np.where(mask, lbl, df_ea['group'])

compare_vars = {}
for col, eng in [('age_at_op',    'Age at surgery'),
                 ('BMI',          'BMI'),
                 ('asascore',     'ASA score'),
                 ('duur_operatie','Op. duration (log)')]:
    if col in df_ea.columns:
        compare_vars[col] = eng

group_order  = list(masks.keys())
group_colors = ['seagreen', 'crimson', 'steelblue', 'coral']
palette_ea   = dict(zip(group_order, group_colors))

n_vars = len(compare_vars)
fig, axes = plt.subplots(1, n_vars, figsize=(3.8 * n_vars, 5))
if n_vars == 1:
    axes = [axes]

for ax, (col, lbl) in zip(axes, compare_vars.items()):
    sns.boxplot(data=df_ea, x='group', y=col, order=group_order,
                palette=palette_ea, ax=ax, width=0.55, fliersize=3)
    ax.set_title(lbl, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Error Analysis: Clinical Variables by Prediction Outcome (RF Stage 4)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()


---
## High-Risk vs Low-Risk Patient Profiles

The 10 patients with the highest and 10 with the lowest predicted complication probability (Stage 4 RF) are compared on key clinical variables.

In [ ]:
# ── High-Risk vs Low-Risk Patient Profiles ────────────────────────────────────
X4_hr, y4_hr = prepare_xy(STAGE4, outcome_col='compl')
oof_rf_hr    = get_oof_probas(X4_hr, y4_hr, 'RF')

df_hr = X4_hr.copy()
df_hr['prob']   = oof_rf_hr
df_hr['actual'] = y4_hr

top10    = df_hr.nlargest(10,  'prob').copy()
bottom10 = df_hr.nsmallest(10, 'prob').copy()
top10['group']    = 'High risk\n(top 10)'
bottom10['group'] = 'Low risk\n(bottom 10)'
combined = pd.concat([top10, bottom10], ignore_index=True)

print("HIGH-RISK  (top 10 predicted probability):")
print(f"  Prob range : {top10['prob'].min():.2f} - {top10['prob'].max():.2f}")
print(f"  Actual complication rate: {top10['actual'].mean():.0%}")
print("\nLOW-RISK  (bottom 10 predicted probability):")
print(f"  Prob range : {bottom10['prob'].min():.2f} - {bottom10['prob'].max():.2f}")
print(f"  Actual complication rate: {bottom10['actual'].mean():.0%}")

profile_vars = {}
for col, eng in [('age_at_op',    'Age at surgery'),
                 ('asascore',     'ASA score'),
                 ('BMI',          'BMI'),
                 ('duur_operatie','Op. duration (log)')]:
    if col in df_hr.columns:
        profile_vars[col] = eng

n_pv = len(profile_vars)
fig, axes = plt.subplots(1, n_pv, figsize=(3.5 * n_pv, 4))
if n_pv == 1:
    axes = [axes]

for ax, (col, lbl) in zip(axes, profile_vars.items()):
    sns.barplot(data=combined, x='group', y=col, ax=ax,
                palette=['crimson', 'steelblue'], capsize=0.1, errorbar='se')
    ax.set_title(lbl, fontweight='bold', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=9)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('High-Risk vs Low-Risk Patient Profiles (Stage 4 RF)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
